# 第十二章：U本位 vs 币本位合约 / Chapter 12: USDT-Margined vs. Coin-Margined Contracts

## 你将学到 / What You Will Learn
- 什么是"U本位"和"币本位" / What "USDT-margined" and "coin-margined" mean
- 两者的盈亏计算有什么不同 / How their P&L math differs
- 为什么币本位"更危险" / Why coin-margined is the more dangerous one
- 什么是"凸性风险" / What "convexity risk" is

---

## 12.1 两种合约的本质区别 / 12.1 The Core Difference Between the Two

|  | U本位 (USDT-Margined) | 币本位 (Coin-Margined) |
|--|----------------------|----------------------|
| 保证金 / Collateral | **USDT**（稳定）/ **USDT** (stable) | **BTC本身** / **BTC itself** |
| 盈亏结算 / Settlement | **USDT** | **BTC** |
| 保证金会缩水？ / Does margin shrink? | **不会** / **No** | **会!**（BTC跌了保证金也跌）/ **Yes!** — if BTC falls, your margin falls too |

## 12.2 关键区别：盈亏曲线的形状 / 12.2 The Key Difference — The Shape of the P&L Curve

**U本位**: 盈亏是一条**直线**（线性）。

**USDT-margined**: P&L is a **straight line** (linear).

**币本位**: 盈亏是一条**弯曲的线**（非线性，这就是"凸性"）。

**Coin-margined**: P&L is a **curved line** (non-linear — this is "convexity").

做多币本位 + BTC暴跌 = **双重打击**：亏钱的同时保证金也在缩水。

Coin-margined long + BTC crash = **double whammy**: you lose money *and* your margin deflates at the same time.

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact, FloatSlider, IntSlider, ToggleButtons
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

C_GREEN, C_RED, C_ORANGE, C_BLUE, C_PURPLE = '#2ecc71', '#e74c3c', '#f39c12', '#3498db', '#9b59b6'
print('Setup done!')

## 12.3 互动对比 / 12.3 Interactive Comparison

In [ ]:
def usdt_vs_coin(entry=60000, leverage=10, qty_btc=1.0, side='Long (buy)'):
    plt.close('all')
    is_long = 'Long' in side
    prices = np.linspace(entry * 0.3, entry * 1.7, 500)

    # USDT-margined
    margin_u = qty_btc * entry / leverage
    pnl_u = qty_btc * (prices - entry) if is_long else qty_btc * (entry - prices)
    liq_u = entry * (1 - 1 / leverage + 0.005) if is_long else entry * (1 + 1 / leverage - 0.005)

    # Coin-margined
    notional = qty_btc * entry
    margin_c_btc = notional / (entry * leverage)
    margin_c_usd = margin_c_btc * entry
    if is_long:
        pnl_c_btc = notional * (1 / entry - 1 / prices)
    else:
        pnl_c_btc = notional * (1 / prices - 1 / entry)
    pnl_c_usd = pnl_c_btc * prices
    liq_c = entry * leverage / (leverage + 1) if is_long else entry * leverage / (leverage - 1)

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # P&L (USD)
    ax = axes[0][0]
    ax.plot(prices, pnl_u, color=C_BLUE, lw=2.5, label='USDT-margined (linear)')
    ax.plot(prices, pnl_c_usd, color=C_ORANGE, lw=2.5, ls='--', label='Coin-margined (convex!)')
    ax.axhline(y=0, color='gray', ls='--')
    ax.axvline(x=entry, color=C_PURPLE, ls=':', alpha=0.5)
    ax.set_xlabel('BTC Price', fontsize=11)
    ax.set_ylabel('P&L (USD)', fontsize=11)
    title_side = 'Long' if is_long else 'Short'
    ax.set_title(f'{title_side}: P&L Comparison', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # ROI
    ax = axes[0][1]
    roi_u = pnl_u / margin_u * 100
    roi_c = np.clip(pnl_c_usd / margin_c_usd * 100, -200, 500) if margin_c_usd > 0 else np.zeros_like(prices)
    ax.plot(prices, roi_u, color=C_BLUE, lw=2, label='USDT ROI')
    ax.plot(prices, roi_c, color=C_ORANGE, lw=2, ls='--', label='Coin ROI')
    ax.axhline(y=0, color='gray', ls='--')
    ax.axhline(y=-100, color=C_RED, ls=':', lw=2, label='-100% (liquidation)')
    ax.set_xlabel('BTC Price', fontsize=11)
    ax.set_ylabel('ROI (%)', fontsize=11)
    ax.set_title('ROI Comparison', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_ylim(-150, 300)
    ax.grid(True, alpha=0.2)

    # Margin value
    ax = axes[1][0]
    ax.plot(prices, np.full_like(prices, margin_u), color=C_BLUE, lw=2.5, label='USDT margin (constant)')
    ax.plot(prices, margin_c_btc * prices, color=C_ORANGE, lw=2.5, ls='--', label='Coin margin (varies!)')
    ax.axvline(x=entry, color=C_PURPLE, ls=':', alpha=0.3)
    ax.fill_between(prices, margin_c_btc * prices, margin_u,
                    where=(margin_c_btc * prices < margin_u), color=C_RED, alpha=0.15)
    ax.set_xlabel('BTC Price', fontsize=11)
    ax.set_ylabel('Margin Value (USD)', fontsize=11)
    ax.set_title('Margin Value vs Price', fontsize=13, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.2)

    # Summary panel
    ax = axes[1][1]
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.axis('off')
    ax.set_title('Key Parameters', fontsize=13, fontweight='bold')
    rows = [
        ('', 'USDT-Margined', 'Coin-Margined', 'black', 'black'),
        ('Margin', f'${margin_u:,.0f} USDT', f'{margin_c_btc:.6f} BTC', C_BLUE, C_ORANGE),
        ('Margin stability', 'Constant', 'Varies with price!', C_BLUE, C_RED),
        ('P&L curve', 'Linear', 'Convex (curved)', C_BLUE, C_ORANGE),
        ('Liq price', f'${liq_u:,.0f}', f'${liq_c:,.0f}', C_BLUE, C_ORANGE),
        ('Risk', 'Normal', 'Double whammy!', C_BLUE, C_RED),
    ]
    for i, (label, v1, v2, c1, c2) in enumerate(rows):
        y = 9 - i * 1.4
        ax.text(0.3, y, label, fontsize=9, fontweight='bold', color='gray')
        ax.text(4.5, y, v1, fontsize=9, color=c1, ha='center')
        ax.text(8.5, y, v2, fontsize=9, color=c2, ha='center')

    plt.tight_layout()
    plt.show()

interact(usdt_vs_coin,
         entry=IntSlider(value=60000, min=20000, max=100000, step=1000, description='Entry($):'),
         leverage=IntSlider(value=10, min=2, max=125, step=1, description='Leverage:'),
         qty_btc=FloatSlider(value=1.0, min=0.1, max=10, step=0.1, description='Qty(BTC):'),
         side=ToggleButtons(options=['Long (buy)', 'Short (sell)'], description='Side:'));

## 12.4 小结 / 12.4 Chapter Recap

> **U本位 / USDT-margined** = 用稳定币做保证金，盈亏是直线 → 风险可预测
> Stablecoin collateral, linear P&L → predictable risk.
>
> **币本位 / Coin-margined** = 用BTC做保证金，盈亏是弯的 → 做多更容易爆仓（双重打击）
> Crypto collateral, convex P&L → long positions liquidate much more easily (double whammy).
>
> **新手建议: 先用U本位 / Beginner advice: start with USDT-margined.**

> **下一章：极端行情模拟** —— BTC暴跌50%时，不同杠杆会怎样？
>
> **Next chapter: crash simulation** — what happens to different leverage levels when BTC drops 50 %?